# Natural Language Processing - Text Preprocessing

## Libraries and settings

In [1]:
# Libraries
import os
import re
import string
import numpy as np
import pandas as pd
from pprint import pprint

import nltk

# Import only once
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.chunk import tree2conlltags
from nltk.chunk import conlltags2tree
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...


Current working directory: /workspaces/data_analytics/Week_11


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


## Defining documents

In [16]:
# Defining documents (=sentenses)
d1 = 'The car is driven on the road.'
d2 = 'The truck is driven on the highway.'
d3 = 'The bicycle is driven on the bicycle path.'
d4 = 'The motorcycle is driven on the street.'
d5 = 'The bus is driven on the avenue.'

corpus_01 = d1 + ' ' + d2 + ' ' + d3 + ' ' + d4 + ' ' + d5
corpus_01

'The car is driven on the road. The truck is driven on the highway. The bicycle is driven on the bicycle path. The motorcycle is driven on the street. The bus is driven on the avenue.'

## Text preprocessing
#### Steps:
- Text to lowercase
- Removing punctuations
- Tokenization
- Removal of stop words
- Lemmatization

### Text to lowercase

In [17]:
# Text to lowercase function
def text_lowercase(text):
    return text.lower()

# Text to lowercase
corpus_02 = text_lowercase(corpus_01)
corpus_02

'the car is driven on the road. the truck is driven on the highway. the bicycle is driven on the bicycle path. the motorcycle is driven on the street. the bus is driven on the avenue.'

### Removing punctuation

In [18]:
# Remove punctuation function
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
corpus_03 = remove_punctuation(corpus_02)
corpus_03

'the car is driven on the road the truck is driven on the highway the bicycle is driven on the bicycle path the motorcycle is driven on the street the bus is driven on the avenue'

### Tokenize text & removal of stopwords

In [19]:
# Show english stopwords
eng_stopwords = set(stopwords.words('english'))
print("List of english stopwords:")
print(eng_stopwords)

List of english stopwords:
{'again', 'which', 'same', "wouldn't", 'from', 'was', "didn't", 'on', "weren't", 'my', 'such', 'will', 'itself', 'no', 'were', 'while', 'yours', 'all', 'is', 'ain', 'few', 'hadn', "we're", 'between', 'ourselves', 'that', "isn't", 'by', 'where', 'as', "that'll", 'had', 'shouldn', 'should', 'here', 'own', "shan't", 'if', "mightn't", 'nor', 'over', 'both', 'about', 'not', 'd', 'an', 'shan', 'wouldn', 'him', "we've", "you'd", "couldn't", "don't", 'y', "she's", 'being', 'themselves', 're', 'for', 'they', 'me', "aren't", "i've", 'each', 'our', 'then', 'couldn', 'only', "we'll", 'why', 'mightn', 'off', 'aren', 'having', 'some', 'are', 'of', 'mustn', 'any', 'wasn', 'herself', 'himself', 'under', 'out', 'yourselves', 'theirs', "i'll", 'o', 'am', 's', 'be', 'than', 'too', "should've", 'now', 'its', "he'd", 'during', 'with', 'been', "they'd", 'did', 'do', 'she', 'whom', 'these', 'your', 'ours', 'it', 'what', 'does', "they've", 'against', 'myself', 'through', "you've", '

In [20]:
# Function for tokenization and the removal of stopwords
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return filtered_text
 
# Remove stopwords
corpus_04 = remove_stopwords(corpus_03)
print(corpus_04, end="")

['car', 'driven', 'road', 'truck', 'driven', 'highway', 'bicycle', 'driven', 'bicycle', 'path', 'motorcycle', 'driven', 'street', 'bus', 'driven', 'avenue']

### Lemmatization

In [21]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize string function
def lemmatize_word(text):
    word_tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word, pos ='v') for word in word_tokens]
    return lemmas

# Lemmatize
lem = []
for i in corpus_04:
    lem.append(lemmatize_word(i))

# Nested list to list
corpus_05 = [' '.join([str(x) for x in lst]) for lst in lem]

print('Before lemmatization:')
print(corpus_04, '\n')

print('After lemmatization:')
print(corpus_05, end="")

Before lemmatization:
['car', 'driven', 'road', 'truck', 'driven', 'highway', 'bicycle', 'driven', 'bicycle', 'path', 'motorcycle', 'driven', 'street', 'bus', 'driven', 'avenue'] 

After lemmatization:
['car', 'drive', 'road', 'truck', 'drive', 'highway', 'bicycle', 'drive', 'bicycle', 'path', 'motorcycle', 'drive', 'street', 'bus', 'drive', 'avenue']

## Redefine the text corpus (pre-processed)

In [22]:
# We will use the lemmatized words above to re-define our corpus 
corpus = ['car drive road', 
          'truck drive highway', 
          'bicycle drive bicycle path',
          'motorcycle drive street',
          'bus drive avenue']

## Document-term matrix with ngram_range=(1,1)

In [23]:
# Vectorizer with ngram_range=(1,1)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(1,1))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   avenue  bicycle  bus  car  drive  highway  motorcycle  path  road  street  \
0       0        0    0    1      1        0           0     0     1       0   
1       0        0    0    0      1        1           0     0     0       0   
2       0        2    0    0      1        0           0     1     0       0   
3       0        0    0    0      1        0           1     0     0       1   
4       1        0    1    0      1        0           0     0     0       0   

   truck  
0      0  
1      1  
2      0  
3      0  
4      0  


## Document-term matrix with ngram_range=(2,2)

In [24]:
# Vectorizer with with ngram_range=(2,2)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(2,2))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   bicycle drive  bicycle path  bus drive  car drive  drive avenue  \
0              0             0          0          1             0   
1              0             0          0          0             0   
2              1             1          0          0             0   
3              0             0          0          0             0   
4              0             0          1          0             1   

   drive bicycle  drive highway  drive road  drive street  motorcycle drive  \
0              0              0           1             0                 0   
1              0              1           0             0                 0   
2              1              0           0             0                 0   
3              0              0           0             1                 1   
4              0              0           0             0                 0   

   truck drive  
0            0  
1            1  
2            0  
3            0 

## Term frequency-inverse document frequency (TF-IDF)
- For details see: https://www.learndatasci.com/glossary/tf-idf-term-frequency-inverse-document-frequency

### Term Frequency (TF)

In [25]:
# Compute Term Frequency (TF)
words_set = set()
for doc in corpus:
    words = doc.split(' ')
    words_set = words_set.union(set(words))
    
print('Number of words in the corpus:',len(words_set), '\n')
print('The words in the corpus: \n', words_set)

# Number of documents in the corpus
n_docs = len(corpus)

# Number of unique words in the corpus 
n_words_set = len(words_set)

df_tf = pd.DataFrame(np.zeros((n_docs, n_words_set)), 
                     columns=list(words_set))

print("\nTerm Frequency (TF):")
for i in range(n_docs):
    # Words in the document
    words = corpus[i].split(' ')
    for w in words:
        df_tf[w][i] = df_tf[w][i] + (1 / len(words))
        
print(df_tf.round(4))

Number of words in the corpus: 11 

The words in the corpus: 
 {'truck', 'bicycle', 'avenue', 'street', 'road', 'drive', 'bus', 'highway', 'path', 'motorcycle', 'car'}

Term Frequency (TF):
    truck  bicycle  avenue  street    road   drive     bus  highway  path  \
0  0.0000      0.0  0.0000  0.0000  0.3333  0.3333  0.0000   0.0000  0.00   
1  0.3333      0.0  0.0000  0.0000  0.0000  0.3333  0.0000   0.3333  0.00   
2  0.0000      0.5  0.0000  0.0000  0.0000  0.2500  0.0000   0.0000  0.25   
3  0.0000      0.0  0.0000  0.3333  0.0000  0.3333  0.0000   0.0000  0.00   
4  0.0000      0.0  0.3333  0.0000  0.0000  0.3333  0.3333   0.0000  0.00   

   motorcycle     car  
0      0.0000  0.3333  
1      0.0000  0.0000  
2      0.0000  0.0000  
3      0.3333  0.0000  
4      0.0000  0.0000  


### Inverse Document Frequency (IDF)

In [26]:
# Computing Inverse Document Frequency (IDF)
print("\nInverse Document Frequency (IDF):")

idf = {}

for w in words_set:
    
    # k = number of documents that contain this word
    k = 0
    
    for i in range(n_docs):
        if w in corpus[i].split():
            k += 1
            
    idf[w] =  np.log10(n_docs / k).round(4)
    
    print(f'{w:>15}: {idf[w]:>10}')


Inverse Document Frequency (IDF):
          truck:      0.699
        bicycle:      0.699
         avenue:      0.699
         street:      0.699
           road:      0.699
          drive:        0.0
            bus:      0.699
        highway:      0.699
           path:      0.699
     motorcycle:      0.699
            car:      0.699


### Term Frequency - Inverse Document Frequency (TF-IDF)

In [27]:
# Computing TF-IDF
df_tf_idf = df_tf.copy()

for w in words_set:
    for i in range(n_docs):
        df_tf_idf[w][i] = df_tf[w][i] * idf[w]

print('\nTF-IDF:')
print(df_tf_idf.round(4))


TF-IDF:
   truck  bicycle  avenue  street   road  drive    bus  highway    path  \
0  0.000   0.0000   0.000   0.000  0.233    0.0  0.000    0.000  0.0000   
1  0.233   0.0000   0.000   0.000  0.000    0.0  0.000    0.233  0.0000   
2  0.000   0.3495   0.000   0.000  0.000    0.0  0.000    0.000  0.1748   
3  0.000   0.0000   0.000   0.233  0.000    0.0  0.000    0.000  0.0000   
4  0.000   0.0000   0.233   0.000  0.000    0.0  0.233    0.000  0.0000   

   motorcycle    car  
0       0.000  0.233  
1       0.000  0.000  
2       0.000  0.000  
3       0.233  0.000  
4       0.000  0.000  


## Part-of-Speach (POS) tagging
For meaning of POS-tags see: https://pythonexamples.org/nltk-pos-tagging

In [28]:
text = '''European authorities fined Google a record $5.1 
          billion on Wednesday for abusing its power in the 
          mobile phone market.'''

def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN>}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

# Print the POS-tags
pprint(iob_tagged)

[('European', 'JJ', 'O'),
 ('authorities', 'NNS', 'O'),
 ('fined', 'VBD', 'O'),
 ('Google', 'NNP', 'O'),
 ('a', 'DT', 'B-NP'),
 ('record', 'NN', 'I-NP'),
 ('$', '$', 'O'),
 ('5.1', 'CD', 'O'),
 ('billion', 'CD', 'O'),
 ('on', 'IN', 'O'),
 ('Wednesday', 'NNP', 'O'),
 ('for', 'IN', 'O'),
 ('abusing', 'VBG', 'O'),
 ('its', 'PRP$', 'O'),
 ('power', 'NN', 'B-NP'),
 ('in', 'IN', 'O'),
 ('the', 'DT', 'B-NP'),
 ('mobile', 'JJ', 'I-NP'),
 ('phone', 'NN', 'I-NP'),
 ('market', 'NN', 'B-NP'),
 ('.', '.', 'O')]


In [30]:
text = """Swiss tech startups are growing rapidly, 
and many international companies are opening new offices in Zurich."""

def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN.*>+}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

pprint(sent)


[('Swiss', 'JJ'),
 ('tech', 'NN'),
 ('startups', 'NNS'),
 ('are', 'VBP'),
 ('growing', 'VBG'),
 ('rapidly', 'RB'),
 (',', ','),
 ('and', 'CC'),
 ('many', 'JJ'),
 ('international', 'JJ'),
 ('companies', 'NNS'),
 ('are', 'VBP'),
 ('opening', 'VBG'),
 ('new', 'JJ'),
 ('offices', 'NNS'),
 ('in', 'IN'),
 ('Zurich', 'NNP'),
 ('.', '.')]


NN (Noun, singular) – names a single thing or concept.
Example from the text: tech (NN).

NNS (Noun, plural) – names more than one thing.
Example: startups, companies, offices (NNS).

JJ (Adjective) – describes or modifies a noun.
Example: Swiss, many, international, new (JJ).

VBP (Verb, non-3rd person singular present) – present-tense verb for subjects I/we/you/they.
Example: are (VBP).

VBG (Verb, gerund or present participle) – “-ing” form of a verb.
Example: growing, opening (VBG).

RB (Adverb) – modifies a verb, adjective or another adverb.
Example: rapidly (RB).

NNP (Proper noun, singular) – name of a specific person, place or organization.
Example: Zurich (NNP).

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1030-azure
Datetime: 2025-11-30 15:21:06
Python Version: 3.11.13
-----------------------------------
